In [35]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

def load_swap_report(path):
    df = pd.read_csv(path)
    df = df.dropna(how='all')  # file has trailing blank padding rows

    # Subscriber/serial numbers come in as floats (e.g. 16421500.0) -> clean ints
    for col in ['Subscriber Number', 'Old Serial Number', 'New Serial Number']:
        df[col] = df[col].astype('int64')

    # Swap Date is "D/M/YYYY 0:00" (time is always 0:00 there, real time is in Swap Time)
    date_part = df['Swap Date'].str.split(' ', n=1).str[0]
    df['SwapDateTime'] = pd.to_datetime(
        date_part + ' ' + df['Swap Time'],
        format='%d/%m/%Y %I:%M:%S %p'
    )
    df = df.loc[df['Item']!='ART Smart Card']
    return df


def build_customer_timeline(df):
    """
    One row per swap event, showing the customer's CURRENT decoder and
    smartcard serial number as of that swap (the item that was swapped
    is updated, the other item is carried forward from its last known value).
    OldDecoder/OldSmartcard show what was replaced IN THAT TRANSACTION only
    (not carried forward) -- blank when that row's swap was for the other item.
    """
    df = df.sort_values(['Subscriber Number', 'SwapDateTime']).copy()

    # New Serial Number becomes the "current" value for whichever Item was swapped
    df['Decoder'] = df['New Serial Number'].where(df['Item'] == 'Decoder')
    df['Smartcard'] = df['New Serial Number'].where(df['Item'] == 'Smartcard')

    # carry forward the last known serial per customer for the item NOT swapped that row
    df[['Decoder', 'Smartcard']] = (
        df.groupby('Subscriber Number')[['Decoder', 'Smartcard']].ffill()
    )

    # transaction-specific old values -- no carry-forward
    df['OldDecoder'] = df['Old Serial Number'].where(df['Item'] == 'Decoder')
    df['OldSmartcard'] = df['Old Serial Number'].where(df['Item'] == 'Smartcard')

    timeline = df[[
        'SwapDateTime', 'Subscriber Number', 'Decoder', 'Smartcard', 'OldDecoder', 'OldSmartcard'
    ]].rename(columns={
        'SwapDateTime': 'Date',
        'Subscriber Number': 'CustomerNumber'
    }).reset_index(drop=True)

    # nullable Int64 so unknown/not-applicable values show as <NA> instead of NaN/float
    for col in ['Decoder', 'Smartcard', 'OldDecoder', 'OldSmartcard']:
        timeline[col] = timeline[col].astype('Int64')

    timeline = drop_incomplete_duplicates(timeline)

    return timeline


def drop_incomplete_duplicates(timeline):
    """
    When the same customer has two rows at the exact same Date (a tie between
    a Decoder swap and a Smartcard swap logged at the same timestamp), the
    ffill step produces one fully-populated row and one still-partial row for
    that instant. Drop the partial one when a full row exists for that
    (CustomerNumber, Date); otherwise leave rows as-is (nothing to drop).
    """
    is_full = timeline[['Decoder', 'Smartcard']].notna().all(axis=1)
    group_has_full = is_full.groupby(
        [timeline['CustomerNumber'], timeline['Date']]
    ).transform('any')
    return timeline[is_full | ~group_has_full].reset_index(drop=True)


def current_state(timeline):
    """Latest known decoder/smartcard per customer (as of the last swap on file)."""
    return timeline.sort_values('Date').groupby('CustomerNumber').tail(1).reset_index(drop=True)


def state_as_of(timeline, as_of_date, customer_number=None):
    """What a customer's decoder/smartcard was at a specific point in time."""
    snap = timeline[timeline['Date'] <= pd.Timestamp(as_of_date)]
    if customer_number is not None:
        snap = snap[snap['CustomerNumber'] == customer_number]
    return snap.sort_values('Date').groupby('CustomerNumber').tail(1).reset_index(drop=True)




if __name__ == '__main__':
    raw = load_swap_report(r"S:\Sheikh\30592658_SWAPRPT.CSV")
    timeline = build_customer_timeline(raw)

    print(timeline.shape)
  

    timeline = drop_incomplete_duplicates(timeline)
    #timeline['OldHardware'] = timeline['OldDecoder'].combine_first(timeline['OldSmartcard'])
    timeline['OldDecoder'] = timeline['OldDecoder'].fillna(timeline.groupby('CustomerNumber')['OldDecoder'].shift(-1))   
    timeline['OldSmartcard'] = timeline['OldSmartcard'].fillna(timeline.groupby('CustomerNumber')['OldSmartcard'].shift(-1))   

    # timeline['Decoder'].fillna(timeline['OldHardware'])
    # timeline['Smartcard'].fillna(timeline['OldHardware'])

    # timeline['temp'] = timeline['OldHardware'].shift(-1)
    # timeline['Smartcard'] = timeline['Smartcard'].fillna(timeline['temp'])
    # timeline = timeline.drop(columns=['OldSmartcard', 'OldDecoder','temp'])

    first_old_sc = (
        timeline.groupby('CustomerNumber')['OldSmartcard']
                .transform(lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA)
    )

    first_old_decoder = (
            timeline.groupby('CustomerNumber')['OldDecoder']
                    .transform(lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA)
        )


    timeline['Smartcard'] = timeline['Smartcard'].fillna(first_old_sc)
    timeline['Decoder'] = timeline['Decoder'].fillna(first_old_decoder)


    display(timeline.loc[timeline['CustomerNumber']==10099993])


    timeline.to_csv('swap_timeline.csv', index = False)

(47888, 6)


,Date,CustomerNumber,Decoder,Smartcard,OldDecoder,OldSmartcard
221,2024-07-08 17:43:11,10099993,319669287,42788934018,314727030,<NA>
222,2025-08-19 14:35:15,10099993,363277515,42788934018,319669287,42788934018
223,2025-11-12 20:24:51,10099993,363277515,10731012059,363277515,42788934018
224,2025-11-12 20:24:51,10099993,349760656,10731012059,363277515,<NA>


In [36]:
raw['Item'].value_counts()

Item
Smartcard    39445
Decoder      23628
Name: count, dtype: int64

In [38]:
raw.loc[raw['Subscriber Number']==1821184]

,Subscriber Number,Subscriber Type,Received Entity,Replacement Number,Item,Old Serial Number,New Serial Number,Swap Date,Swap Time,User Name,SwapDateTime
42000,1821184,beIN Quartar Installment,EDD MADENAT EL SALAM,15448,Smartcard,42916698485,10697335908,23/03/2021 12:00:00 AM,12:30:20 PM,Abeer El Araby,2021-03-23 12:30:20
